<a href="https://colab.research.google.com/github/akul-bharadwaj/various-agents/blob/main/Tool_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool Calling Agent with LangChain

In [ ]:
!pip -q uninstall numpy -y
!pip -q install numpy==1.26.4
!pip -q install langchain==0.2.16 langchain-openai langchain-experimental pandas python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 67.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompat

In [ ]:
import csv

data = [
    ["Employee", "Department", "Salary", "Performance", "Experience", "Country"],
    ["Alice", "Engineering", 95000, "Excellent", 7, "India"],
    ["Bob", "Marketing", 60000, "Average", 3, "USA"],
    ["Carol", "Engineering", 110000, "Excellent", 10, "Germany"],
    ["David", "Sales", 55000, "Poor", 2, "India"],
    ["Eva", "HR", 50000, "Good", 4, "UK"],
    ["Frank", "Engineering", 120000, "Excellent", 12, "USA"],
    ["Grace", "Marketing", 70000, "Good", 5, "Singapore"],
    ["Henry", "Sales", 65000, "Average", 6, "UAE"],
    ["Iris", "HR", 48000, "Poor", 1, "Canada"],
    ["Jack", "Engineering", 105000, "Good", 8, "Japan"],
]

with open("employees.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(data)

print("employees.csv created!")

employees.csv created!


In [ ]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

Enter OpenAI API Key: ··········


In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Tools
@tool
def lowest_performer_department() -> str:
    """Find the department with the most 'Poor' performers."""
    poor_counts = {}
    for row in data[1:]:  # skip header
        if row[3] == "Poor":
            dept = row[1]
            poor_counts[dept] = poor_counts.get(dept, 0) + 1
    if not poor_counts:
        return "No 'Poor' performers found."
    worst_dept = max(poor_counts, key=poor_counts.get)
    return f"{worst_dept} ({poor_counts[worst_dept]} Poor performer(s))"

@tool
def salary_gap(department: str) -> str:
    """Find the salary gap (max - min) within a department."""
    salaries = [
        row[2] for row in data[1:]
        if row[1].lower() == department.lower()
    ]
    if not salaries:
        return f"No employees found in '{department}'."
    gap = max(salaries) - min(salaries)
    return (
        f"Department: {department}\n"
        f"Highest: ${max(salaries):,} | Lowest: ${min(salaries):,}\n"
        f"Salary gap: ${gap:,}"
    )

@tool
def improvement_plan(department: str, gap: int) -> str:
    """Suggest an action plan to fix performance and salary issues."""
    return f"""
Improvement Plan — {department}
================================
Salary Gap: ${gap:,}

Performance Actions:
- Conduct 1:1 reviews with all 'Poor' rated employees.
- Set clear 90-day improvement goals with measurable KPIs.
- Assign mentors from high-performing team members.
- Schedule bi-weekly check-ins to track progress.

Salary Actions:
- Audit compensation vs. market benchmarks for {department}.
- Identify if gap correlates with experience or performance.
- Create a structured pay band to reduce disparity over 2 quarters.
- Introduce performance-linked increments to motivate improvement.
"""

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools whenever necessary."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

tools = [lowest_performer_department, salary_gap, improvement_plan]

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
response = agent_executor.invoke({"input": "Which department has the most poor performers, what is the salary gap in that department, and what is your improvement plan?"})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `lowest_performer_department` with `{}`


Sales (1 Poor performer(s))
Invoking: `salary_gap` with `{'department': 'Sales'}`


Department: Sales
Highest: $65,000 | Lowest: $55,000
Salary gap: $10,000
Invoking: `improvement_plan` with `{'department': 'Sales', 'gap': 10000}`



Improvement Plan — Sales
Salary Gap: $10,000

Performance Actions:
- Conduct 1:1 reviews with all 'Poor' rated employees.
- Set clear 90-day improvement goals with measurable KPIs.
- Assign mentors from high-performing team members.
- Schedule bi-weekly check-ins to track progress.

Salary Actions:
- Audit compensation vs. market benchmarks for Sales.
- Identify if gap correlates with experience or performance.
- Create a structured pay band to reduce disparity over 2 quarters.
- Introduce performance-linked increments to motivate improvement.
The department with the most poor performers is the **Sales** department, which has 1 poor performer. The salary gap in the